In [1]:
import os
import gc
import pandas as pd
import torch

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
)

from peft import PeftModel

from rouge_score import rouge_scorer


# ============================================================
# CONFIG
# ============================================================

BASE_MODEL_NAME = r"../../NLP/models--Qwen--Qwen2.5-1.5B-Instruct/snapshots/989aa7980e4cf806f80c7fef2b1adb7bc71aa306"

FINETUNED_ADAPTER = "./qwen2_5_1_5b_turkish_law_qlora"

DATASET_NAME = "data/renicames_turkish_law_chatbot"

# Set to True to test on all data, or False to limit to NUM_SAMPLES for a quick run
TEST_ALL = True
NUM_SAMPLES = 40

# Batch size for generation. Higher uses more VRAM and runs much faster.
# 64 is extremely fast and fits perfectly under 8GB VRAM (uses ~5.0GB VRAM total).
BATCH_SIZE = 64

SYSTEM_PROMPT = (
    "Sen Türk hukuku konusunda yardımcı olan bir asistansın. "
    "Sorulara kısa, açık ve hukuki doğruluğa dikkat ederek cevap ver."
)


# ============================================================
# DEVICE SETUP
# ============================================================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
if torch.cuda.is_available():
    print("GPU Name:", torch.cuda.get_device_name(0))
    # Make sure PyTorch uses GPU 0 if multiple are available
    os.environ["CUDA_VISIBLE_DEVICES"] = "0"


# ============================================================
# LOAD DATASET
# ============================================================

ds = load_dataset("json", data_files={"test": f"{DATASET_NAME}/test.jsonl"})

if "test" in ds:
    eval_ds = ds["test"]
else:
    eval_ds = ds["train"]

# Slice if not testing all
if not TEST_ALL:
    eval_ds = eval_ds.select(range(min(NUM_SAMPLES, len(eval_ds))))

print(f"Total test samples to evaluate: {len(eval_ds)}")


# ============================================================
# DETECT COLUMNS
# ============================================================

sample = eval_ds[0]

question_col = None
answer_col = None

for c in ["Soru", "soru", "question", "instruction"]:
    if c in sample:
        question_col = c
        break

for c in ["Cevap", "cevap", "answer", "output"]:
    if c in sample:
        answer_col = c
        break

print("Question column:", question_col)
print("Answer column:", answer_col)


# ============================================================
# TOKENIZER
# ============================================================

tokenizer = AutoTokenizer.from_pretrained(
    BASE_MODEL_NAME,
    trust_remote_code=True
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Padding side must be left for decoder-only batched generation
tokenizer.padding_side = "left"


# ============================================================
# GENERATION FUNCTION FOR BATCHED SEQUENTIAL EVALUATION
# ============================================================

def generate_answers_for_model(model_name, adapter_path=None, batch_size=BATCH_SIZE):
    device_map = {"": "cuda:0"} if torch.cuda.is_available() else None
    torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32

    model_label = "FINE-TUNED" if adapter_path else "BASE"
    print(f"\nLoading {model_label} model into memory...")
    
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        device_map=device_map,
        torch_dtype=torch_dtype,
        trust_remote_code=True,
        attn_implementation="sdpa"
    )
    
    if adapter_path is not None:
        print(f"Applying Adapter: {adapter_path}...")
        model = PeftModel.from_pretrained(model, adapter_path)
        if torch.cuda.is_available():
            model = model.to("cuda")
            
    model.eval()
    
    predictions = []
    total = len(eval_ds)
    
    print(f"Generating answers for {model_label} model in batches of {batch_size}...")
    
    # Batch generation
    for i in range(0, total, batch_size):
        batch_rows = eval_ds.select(range(i, min(i + batch_size, total)))
        batch_texts = []
        
        for row in batch_rows:
            question = str(row[question_col])
            messages = [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": question}
            ]
            text = tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True
            )
            batch_texts.append(text)
            
        model_device = next(model.parameters()).device
        inputs = tokenizer(
            batch_texts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=512
        ).to(model_device)
        
        input_length = inputs.input_ids.shape[-1]
        
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=256,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id
            )
            
        for out in outputs:
            generated_tokens = out[input_length:]
            pred = tokenizer.decode(generated_tokens, skip_special_tokens=True)
            predictions.append(pred)
            
        print(f"Progress: [{min(i + batch_size, total)}/{total}] queries generated.")
            
    # Clean up model from GPU memory completely
    del model
    if torch.cuda.is_available():
        gc.collect()
        torch.cuda.empty_cache()
        
    print(f"{model_label} evaluation completed and memory cleared.")
    return predictions


# ============================================================
# GENERATE PREDICTIONS SEQUENTIALLY
# ============================================================

base_preds = generate_answers_for_model(BASE_MODEL_NAME)
ft_preds = generate_answers_for_model(BASE_MODEL_NAME, FINETUNED_ADAPTER)


# ============================================================
# METRICS
# ============================================================

rouge = rouge_scorer.RougeScorer(
    ["rouge1", "rouge2", "rougeL"],
    use_stemmer=False
)


def token_f1(reference, prediction):

    ref_tokens = reference.split()
    pred_tokens = prediction.split()

    common = set(ref_tokens) & set(pred_tokens)

    if len(common) == 0:
        return 0.0

    precision = len(common) / max(len(pred_tokens), 1)
    recall = len(common) / max(len(ref_tokens), 1)

    return 2 * precision * recall / (precision + recall)


# ============================================================
# EVALUATION
# ============================================================

results = []
total_samples = len(eval_ds)

print("\nComputing metrics and printing predictions...")
for i, row in enumerate(eval_ds):
    question = str(row[question_col])
    reference = str(row[answer_col])
    
    base_pred = base_preds[i]
    ft_pred = ft_preds[i]

    # ROUGE
    base_scores = rouge.score(reference, base_pred)
    ft_scores = rouge.score(reference, ft_pred)

    # TOKEN F1
    base_f1 = token_f1(reference, base_pred)
    ft_f1 = token_f1(reference, ft_pred)

    results.append({
        "question": question,
        "reference": reference,
        "base_prediction": base_pred,
        "finetuned_prediction": ft_pred,
        "base_rouge1": base_scores["rouge1"].fmeasure,
        "base_rouge2": base_scores["rouge2"].fmeasure,
        "base_rougeL": base_scores["rougeL"].fmeasure,
        "base_f1": base_f1,
        "ft_rouge1": ft_scores["rouge1"].fmeasure,
        "ft_rouge2": ft_scores["rouge2"].fmeasure,
        "ft_rougeL": ft_scores["rougeL"].fmeasure,
        "ft_f1": ft_f1,
    })

    # Only print details for the first 10 to keep log clean
    if i < 10:
        print(f"\n[{i+1}/{total_samples}]")
        print("Question:", question[:100])
        print("\nBASE:")
        print(base_pred[:300])
        print("\nFINE-TUNED:")
        print(ft_pred[:300])


# ============================================================
# SAVE RESULTS
# ============================================================

df = pd.DataFrame(results)

df.to_csv(
    "qwen_base_vs_finetuned.csv",
    index=False
)

print("\nSaved results.")


# ============================================================
# SUMMARY
# ============================================================

print("\n========== SUMMARY ==========")

print("\nBASE MODEL")
print("ROUGE-1:", df["base_rouge1"].mean())
print("ROUGE-2:", df["base_rouge2"].mean())
print("ROUGE-L:", df["base_rougeL"].mean())
print("Token F1:", df["base_f1"].mean())

print("\nFINE-TUNED MODEL")
print("ROUGE-1:", df["ft_rouge1"].mean())
print("ROUGE-2:", df["ft_rouge2"].mean())
print("ROUGE-L:", df["ft_rougeL"].mean())
print("Token F1:", df["ft_f1"].mean())


Using device: cuda
GPU Name: NVIDIA GeForce RTX 4060 Laptop GPU
Total test samples to evaluate: 1500
Question column: Soru
Answer column: Cevap


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!



Loading BASE model into memory...


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Generating answers for BASE model in batches of 64...
Progress: [64/1500] queries generated.
Progress: [128/1500] queries generated.
Progress: [192/1500] queries generated.
Progress: [256/1500] queries generated.
Progress: [320/1500] queries generated.
Progress: [384/1500] queries generated.
Progress: [448/1500] queries generated.
Progress: [512/1500] queries generated.
Progress: [576/1500] queries generated.
Progress: [640/1500] queries generated.
Progress: [704/1500] queries generated.
Progress: [768/1500] queries generated.
Progress: [832/1500] queries generated.
Progress: [896/1500] queries generated.
Progress: [960/1500] queries generated.
Progress: [1024/1500] queries generated.
Progress: [1088/1500] queries generated.
Progress: [1152/1500] queries generated.
Progress: [1216/1500] queries generated.
Progress: [1280/1500] queries generated.
Progress: [1344/1500] queries generated.
Progress: [1408/1500] queries generated.
Progress: [1472/1500] queries generated.
Progress: [1500/150

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Applying Adapter: ./qwen2_5_1_5b_turkish_law_qlora...


W0522 18:17:58.590000 29036 torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels


Generating answers for FINE-TUNED model in batches of 64...
Progress: [64/1500] queries generated.
Progress: [128/1500] queries generated.
Progress: [192/1500] queries generated.
Progress: [256/1500] queries generated.
Progress: [320/1500] queries generated.
Progress: [384/1500] queries generated.
Progress: [448/1500] queries generated.
Progress: [512/1500] queries generated.
Progress: [576/1500] queries generated.
Progress: [640/1500] queries generated.
Progress: [704/1500] queries generated.
Progress: [768/1500] queries generated.
Progress: [832/1500] queries generated.
Progress: [896/1500] queries generated.
Progress: [960/1500] queries generated.
Progress: [1024/1500] queries generated.
Progress: [1088/1500] queries generated.
Progress: [1152/1500] queries generated.
Progress: [1216/1500] queries generated.
Progress: [1280/1500] queries generated.
Progress: [1344/1500] queries generated.
Progress: [1408/1500] queries generated.
Progress: [1472/1500] queries generated.
Progress: [15